In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from google.colab import files
uploaded = files.upload("heart.csv")
df = pd.read_csv("heart.csv/heart.csv")

nulls = df.isnull().sum()

if nulls.sum() > 0:

    if nulls['trestbps'] > 0:
        df['trestbps'] = df['trestbps'].fillna(df['trestbps'].mean())

    if nulls['chol'] > 0:
        df['chol'] = df['chol'].fillna(df['chol'].mean())

    if nulls['fbs'] > 0:
        df['fbs'] = df['fbs'].fillna(df['fbs'].mode()[0])

    if nulls['exang'] > 0:
        df['exang'] = df['exang'].fillna(df['exang'].mode()[0])

    if nulls['restecg'] > 0:
        df['restecg'] = df['restecg'].fillna(df['restecg'].mode()[0])

if df.duplicated().sum() > 0:
    df = df.drop_duplicates()

df['fbs'] = df['fbs'].map({True: 1, False: 0})
df['exang'] = df['exang'].map({True: 1, False: 0})

df['restecg'] = df['restecg'].replace('normal', 0)
df['restecg'] = df['restecg'].replace('lv hypertrophy', 1)
df['restecg'] = df['restecg'].replace('st-t abnormality', 2)

encoder = OneHotEncoder(sparse_output=False)

sex_encoded = encoder.fit_transform(df[['sex']])
sex_df = pd.DataFrame(sex_encoded, columns=encoder.get_feature_names_out(['sex']))

df = df.drop('sex', axis=1)
df = pd.concat([df, sex_df], axis=1)

df = df[['age', 'trestbps', 'chol', 'sex_0', 'sex_1']]

for col in ['age', 'trestbps', 'chol']:

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_limit = Q1 - 1.5 * IQR
    upper_limit = Q3 + 1.5 * IQR

    outliers = df[(df[col] < lower_limit) | (df[col] > upper_limit)]

    if len(outliers) > 0:
        df = df[(df[col] >= lower_limit) & (df[col] <= upper_limit)]

scaler = MinMaxScaler()
df[['age', 'trestbps', 'chol']] = scaler.fit_transform(df[['age', 'trestbps', 'chol']])

X = df[['age']]
y = df[['trestbps', 'chol']]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

df.to_csv("heart_cleaned_final.csv", index=False)

Saving heart.csv to heart.csv/heart (2).csv


In [ ]:

cleaned_df = pd.read_csv('heart_cleaned_final.csv')
display(cleaned_df.head())

,age,trestbps,chol,sex_0,sex_1
0,0.479167,0.407895,0.367521,0.0,1.0
1,0.500000,0.605263,0.329060,0.0,1.0
2,0.854167,0.671053,0.205128,0.0,1.0
3,0.666667,0.710526,0.329060,0.0,1.0
4,0.687500,0.578947,0.717949,1.0,0.0


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from google.colab import files

try:
    with open('heart.csv', 'r') as f:
        print("'heart.csv' found.")
except FileNotFoundError:
    print("file not found")
    uploaded = files.upload()
    if 'heart.csv' not in uploaded:
        raise FileNotFoundError("heart.csv was not uploaded.")
df = pd.read_csv("heart.csv")

nulls = df.isnull().sum()

if nulls.sum() > 0:
    if nulls['trestbps'] > 0:
        df['trestbps'] = df['trestbps'].fillna(df['trestbps'].mean())
    if nulls['chol'] > 0:
        df['chol'] = df['chol'].fillna(df['chol'].mean())
    if nulls['fbs'] > 0:
        df['fbs'] = df['fbs'].fillna(df['fbs'].mode()[0])
    if nulls['exang'] > 0:
        df['exang'] = df['exang'].fillna(df['exang'].mode()[0])
    if nulls['restecg'] > 0:
        df['restecg'] = df['restecg'].fillna(df['restecg'].mode()[0])

if df.duplicated().sum() > 0:
    df = df.drop_duplicates()

df['fbs'] = df['fbs'].map({True: 1, False: 0})
df['exang'] = df['exang'].map({True: 1, False: 0})

df['restecg'] = df['restecg'].replace('normal', 0)
df['restecg'] = df['restecg'].replace('lv hypertrophy', 1)
df['restecg'] = df['restecg'].replace('st-t abnormality', 2)

encoder = OneHotEncoder(sparse_output=False)
sex_encoded = encoder.fit_transform(df[['sex']])
sex_df = pd.DataFrame(sex_encoded, columns=encoder.get_feature_names_out(['sex']))

df = df.drop('sex', axis=1)
df = pd.concat([df, sex_df], axis=1)

df = df[['age', 'trestbps', 'chol', 'sex_0', 'sex_1']]

for col in ['age', 'trestbps', 'chol']:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_limit = Q1 - 1.5 * IQR
    upper_limit = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower_limit) | (df[col] > upper_limit)]
    if len(outliers) > 0:
        df = df[(df[col] >= lower_limit) & (df[col] <= upper_limit)]

scaler = MinMaxScaler()
df[['age', 'trestbps', 'chol']] = scaler.fit_transform(df[['age', 'trestbps', 'chol']])

df.to_csv("heart_cleaned_final.csv", index=False)
print("'heart_cleaned_final.csv' created successfully.")

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,confusion_matrix

cleaned_df = pd.read_csv('heart_cleaned_final.csv')

x=cleaned_df[['age', 'trestbps', 'chol']]
y=cleaned_df['sex_1']
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)
train_data = x_train.copy()
train_data['target'] = y_train
train_data.dropna(inplace=True)
x_train = train_data.drop('target', axis=1)
y_train = train_data['target']
test_data = x_test.copy()
test_data['target'] = y_test
test_data.dropna(inplace=True)
x_test = test_data.drop('target', axis=1)
y_test = test_data['target']
model=LogisticRegression(max_iter=1000)
model.fit(x_train,y_train)
y_pred=model.predict(x_test)
acc=accuracy_score(y_test,y_pred)
print("accuracy  ",acc)
cm=confusion_matrix(y_test,y_pred)
print("confusion matrix  ",cm)

Attempting to upload 'heart.csv' if not already present...
'heart.csv' found.
Loading and cleaning 'heart.csv' to create 'heart_cleaned_final.csv'...
'heart_cleaned_final.csv' created successfully.
accuracy   0.7560975609756098
confusion matrix   [[ 0 10]
 [ 0 31]]
